<a href="https://colab.research.google.com/github/eduardobbastos/colabs/blob/main/Hermes_Ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Pipeline Hermes AI - Processamento Automatizado de Poços

Este notebook executa toda a esteira de engenharia de dados para perfis geofísicos (DLIS e LAS).

### 🚀 Etapas do Pipeline:
1.  **Ambiente & Credenciais**: Setup híbrido (Colab/Local) com gestão segura de chaves.
2.  **Ingestão Inteligente**: Download de fontes externas (URLs, Nextcloud) com suporte a descompactação automática.
3.  **Processamento Avançado (`dlisio` & `lasio`)**:
    *   Extração robusta de metadados e alinhamento de curvas por profundidade.
4.  **Geração de Produtos (Multi-Output)**: `_meta.txt`, `_data.csv`, `_plot.png`.

### ☁️ Importância do Google Drive & Requisitos
*   **Por que Google Drive?**: Ele atua como seu **"Datalake" persistente**. Ao contrário da máquina virtual do Colab (que apaga tudo ao reiniciar), o Drive garante que seus dados processados e relatórios fiquem salvos para sempre.
*   **O que é necessário?**: Para o pipeline funcionar, ele precisa de permissão de acesso. O script gerencia isso automaticamente, mas certifique-se de ter aceitado a montagem do Drive quando solicitado.

---

In [ ]:
# @title ⚙️ 1. Inicialização do Sistema (Execute esta célula uma vez)
# --- Instalação & Imports ---
# Instalação silenciosa
# --- Configuração de Logs e Silêncio ---
import logging
import sys

# Configura Logger Global
logger = logging.getLogger('HermesPipeline')
logger.setLevel(logging.INFO)

# Evita duplicidade de handlers
if not logger.handlers:
    # File Handler (Salva em arquivo)
    # O caminho será definido dinamicamente no setup, mas iniciamos com um padrão
    # Para garantir silencio no console, NÃO adicionamos StreamHandler (stdout) se não solicitado
    pass

def setup_logger(log_dir):
    LOG_FILE = os.path.join(log_dir, f"pipeline_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
    fh = logging.FileHandler(LOG_FILE)
    formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
    fh.setFormatter(formatter)
    logger.addHandler(fh)
    print(f"\n[LOG] Saída redirecionada para: {LOG_FILE}")
    return LOG_FILE

def log(msg, silence=True):
    """Registra no log e opcionalmente no console."""
    logger.info(msg)
    if not silence:
        log(msg)

!pip install dlisio lasio openpyxl matplotlib pandas requests -q

import os
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dlisio
import lasio
import shutil
import json
from io import BytesIO
from urllib.parse import urlparse, unquote
from google.colab import drive
from google.colab import files
from datetime import datetime

# Configuração visual
plt.style.use('seaborn-v0_8-whitegrid')
log("Bibliotecas instaladas com sucesso.")


# ------------------------------

def setup_workspace_with_credentials(project_name="Engenharia_Dados_Poco", local_cred_path=None):
    """
    Configura o ambiente de trabalho.
    - Tenta montar o Google Drive (Colab).
    - Se falhar, assume execução LOCAL e usa diretórios locais.
    - Gerencia credenciais.
    """
    is_colab = False
    base_path = ""

    # 1. Tenta montar o Drive
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive'):
            log("Tentando montar Google Drive...")
            drive.mount('/content/drive')
        is_colab = True
        base_path = os.path.join('/content/drive/MyDrive', project_name)
    except Exception as e:
        log(f"\n[AVISO] Não foi possível montar o Google Drive ({e}).")
        log("Alternando para MODO LOCAL.")
        # Define caminho base local (pasta atual + projeto)
        base_path = os.path.join(os.getcwd(), project_name)

    # 3. Definir Subdiretórios
    # Define Timestamp da Execução
    run_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_folder = f"Analysis_{run_ts}"

    # 3. Definir Subdiretórios (Com separação por Timestamp)
    dirs = {
        'root': base_path,
        'config': os.path.join(base_path, '00_Config'),
        'input_root': os.path.join(base_path, '01_Raw_Data'),
        'input_run': os.path.join(base_path, '01_Raw_Data', run_folder),
        'input': os.path.join(base_path, '01_Raw_Data'), # Mantem compatibilidade
        'output': os.path.join(base_path, '02_Processed', run_folder),
        'images': os.path.join(base_path, '03_Plots', run_folder)
    }

    # 4. Criar estrutura
    log(f"\nVerificando estrutura de diretórios em: {base_path}")
    for key, path in dirs.items():
        if not os.path.exists(path):
            os.makedirs(path)
            log(f"[CRIADO] {key}: {path}")
        else:
            log(f"[OK] {key}: {path}")

    # 5. Gerenciamento de Credenciais
    # Nome padrão que o script espera na pasta config
    target_cred_path = os.path.join(dirs['config'], "credentials.json")

    # Se fornecido um caminho local específico (Argumento da função)
    if local_cred_path and os.path.exists(local_cred_path):
        log(f"\n[INFO] Usando credencial local fornecida: {local_cred_path}")
        # Copia para a pasta de config do projeto para padronização, se ainda não existir lá
        if not os.path.exists(target_cred_path):
             shutil.copy(local_cred_path, target_cred_path)
             log(f"       -> Copiada para: {target_cred_path}")

    # Verificação Final
    if not os.path.exists(target_cred_path):
        log(f"\n[!] Credencial 'credentials.json' não encontrada em {dirs['config']}.")

        if is_colab:
            log("Por favor, faça o upload do arquivo JSON de credenciais agora.")
            from google.colab import files
            uploaded = files.upload()
            for filename in uploaded.keys():
                shutil.move(filename, target_cred_path)
                log(f"[SUCESSO] Credencial salva em: {target_cred_path}")
                break
        else:
            log(f"[ERRO] Execute localmente copiando o arquivo 'credentials.json' para: {dirs['config']}")
    else:
        log(f"\n[OK] Credenciais validadas em: {target_cred_path}")

    return dirs, target_cred_path

# --- EXECUÇÃO DO SETUP ---
# Caminho local sugerido pelo usuário
caminho_credencial_local = "/home/ecosta/pipeline/google_api_key.json"

PATHS, CREDENTIALS_PATH = setup_workspace_with_credentials(
    "Analise_Poco_Automatizada",
    local_cred_path=caminho_credencial_local
)


# ------------------------------


def get_filename_from_url(url, response):
    """
    Tenta deduzir o nome do arquivo via Header ou URL.
    """
    # 1. Tenta pegar do Content-Disposition (se o servidor informar)
    if "Content-Disposition" in response.headers:
        cd = response.headers["Content-Disposition"]
        if "filename=" in cd:
            return cd.split("filename=")[1].strip('"')

    # 2. Fallback: Pega da URL
    parsed_url = urlparse(url)
    filename = os.path.basename(parsed_url.path)
    return unquote(filename) if filename else "arquivo_desconhecido.dat"

def smart_ingestion(url, destination_folder):
    """
    Baixa o arquivo e decide automaticamente se deve descompactar ou apenas salvar.
    """
    if not os.path.exists(destination_folder):
        os.makedirs(destination_folder)

    # Correção para URLs Nextcloud/CPRM que precisam de /download
    # Se for um link de compartilhamento publico (/s/) e não tiver /download, adiciona.
    if "/s/" in url and "/download" not in url:
         # Verifica padrão simples do nextcloud.
         # Porém, cuidado para não quebrar urls normais.
         # O mais seguro é tentar baixar e ver o content-type ou dispostion, mas vamos seguir a lógica simples validada anteriormente:
         if "reate.cprm.gov.br" in url or "index.php" in url: # Heurística simples
             url_final = url.rstrip('/') + "/download"
         else:
             url_final = url
    else:
         url_final = url

    log(f"\n[INGESTÃO] Conectando a: {url_final}")
    try:
        # Stream download para suportar arquivos grandes sem estourar RAM
        with requests.get(url_final, stream=True) as r:
            r.raise_for_status()

            # Define nome temporário e caminho
            filename = get_filename_from_url(url_final, r)
            # Limpa caracteres ruins do filename
            filename = filename.replace("%20", "_").replace(" ", "_")

            temp_path = os.path.join(destination_folder, f"temp_{filename}")
            final_path = os.path.join(destination_folder, filename)

            # Baixa para um arquivo temporário
            log(f"Baixando: {filename}...")
            with open(temp_path, 'wb') as f:
                shutil.copyfileobj(r.raw, f)

        # Verifica se é um ZIP válido (inspecionando o header do arquivo, não a extensão)
        if zipfile.is_zipfile(temp_path):
            log(f"Detectado arquivo ZIP. Extraindo em: {destination_folder}")
            with zipfile.ZipFile(temp_path, 'r') as z:
                z.extractall(destination_folder)

            # Remove o zip original para economizar espaço (opcional)
            os.remove(temp_path)
            log("Extração concluída e arquivo temporário removido.")

        else:
            # Não é ZIP, renomeia o temporário para o nome final
            log(f"Arquivo não é ZIP. Salvando como: {filename}")
            if os.path.exists(final_path):
                os.remove(final_path) # Sobrescreve se existir
            os.rename(temp_path, final_path)

        # Retorna lista atualizada de arquivos na pasta
        files_found = []
        for root, dirs, filenames in os.walk(destination_folder):
            for f in filenames:
                if f.lower().endswith(('.dlis', '.las', '.lis')):
                    files_found.append(os.path.join(root, f))

        return files_found

    except Exception as e:
        log(f"[ERRO] Falha na ingestão: {e}")
        # Limpeza em caso de erro
        if 'temp_path' in locals() and os.path.exists(temp_path):
            os.remove(temp_path)
        return []



# ------------------------------

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import lasio
import dlisio

def save_metadata_txt(file_base_name, output_dir, metadata_dict, channel_stats_df):
    """Salva relatorio de metadados em TXT."""
    txt_path = os.path.join(output_dir, f"{file_base_name}_meta.txt")
    with open(txt_path, 'w') as f:
        f.write(f"RELATORIO DE METADADOS: {file_base_name}\n")
        f.write("="*50 + "\n\n")

        f.write("[PARAMETROS DO POCO]\n")
        for k, v in metadata_dict.items():
            f.write(f"{k:<10}: {v}\n")
        f.write("\n" + "-"*50 + "\n\n")

        f.write("[ESTATISTICAS DOS CANAIS]\n")
        if not channel_stats_df.empty:
            f.write(channel_stats_df.to_string(index=False))
        else:
            f.write("Nenhum canal encontrado.")
    return txt_path

def save_curves_csv(file_base_name, output_dir, curves_df):
    """Salva dados das curvas em CSV flat."""
    csv_path = os.path.join(output_dir, f"{file_base_name}_data.csv")
    curves_df.to_csv(csv_path, index=True)
    return csv_path

def generate_triple_combo(file_base_name, output_dir, df, curves_map):
    """
    Gera plot Triple Combo se as curvas existirem.
    """
    # Mnemônicos comuns
    gr_curves = ['GR', 'CGR', 'SGR', 'GAM', 'GRC']
    res_curves = ['RT', 'RES', 'LLD', 'LLS', 'AT90', 'AT60', 'ILD', 'ILM']
    neu_curves = ['NPHI', 'TNPH', 'CNPOR']
    den_curves = ['RHOB', 'DEN', 'ZDEN']

    if df.empty: return None

    # Encontra colunas disponíveis no DF
    cols = df.columns.tolist()
    gr = next((c for c in gr_curves if c in cols), None)
    res = next((c for c in res_curves if c in cols), None)
    neu = next((c for c in neu_curves if c in cols), None)
    den = next((c for c in den_curves if c in cols), None)

    if not any([gr, res, neu, den]):
        return None

    fig, ax = plt.subplots(nrows=1, ncols=3, figsize=(12, 10), sharey=True)
    fig.suptitle(f"Triple Combo: {file_base_name}", fontsize=16)

    # Track 1: Gamma Ray
    if gr:
        ax[0].plot(df[gr], df.index, color='green', linewidth=0.8)
        ax[0].set_xlabel(f"{gr} (gAPI)")
        ax[0].set_xlim(0, 150)
        ax[0].grid(True, which='both', linestyle='--', linewidth=0.5)

    # Track 2: Resistivity (Log)
    if res:
        ax[1].plot(df[res], df.index, color='red', linewidth=0.8)
        ax[1].set_xlabel(f"{res} (ohm.m)")
        ax[1].set_xscale('log')
        ax[1].set_xlim(0.2, 2000)
        ax[1].grid(True, which='both', linestyle='--', linewidth=0.5)

    # Track 3: Porosity (Neu/Den)
    if neu:
        ax[2].plot(df[neu], df.index, color='blue', linestyle='--', label=neu)
        ax[2].set_xlim(0.45, -0.15)
    if den:
        ax[2].plot(df[den], df.index, color='red', linewidth=0.8, label=den)

    ax[2].set_xlabel("Porosity / Density")
    ax[2].legend(loc='upper right', fontsize='small')
    ax[2].grid(True, which='both', linestyle='--', linewidth=0.5)

    ax[0].set_ylim(df.index.max(), df.index.min()) # Inverte profundidade
    ax[0].set_ylabel("Depth")

    plot_path = os.path.join(output_dir, f"{file_base_name}_plot.png")
    plt.tight_layout()
    plt.savefig(plot_path, dpi=150)
    plt.close()
    return plot_path

def process_dlis_multi_output(file_path, output_dir, plot_dir=None):
    """Processa DLIS com rigor técnico (dlisio docs)."""
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    log(f"\n>>> Processando DLIS: {file_name}")

    try:
        with dlisio.dlis.load(file_path) as files:
            for f in files:
                # 1. Metadados Globais (Origin/Parameters)
                meta_dict = {}
                desired = ['WN', 'LNAM', 'LAT', 'LONG', 'LATI', 'LONGI', 'TDD', 'TDL', 'PDAT', 'COMP', 'WELL']

                for param in f.parameters:
                    if param.name.upper() in desired:
                        val = param.values
                        val_str = str(val[0]) if (isinstance(val, list) and len(val)>0) else str(val)
                        meta_dict[param.name.upper()] = val_str.replace("'","").replace("[","").replace("]","")

                if 'LATI' in meta_dict and 'LAT' not in meta_dict: meta_dict['LAT'] = meta_dict['LATI']

                # 2. Dados de Canal (Por Frame)
                # A norma DLIS organiza dados em Frames. Cada frame tem um Índice (Depth/Time).
                if f.frames:
                    for frame in f.frames:
                        curves_data = {}
                        stats_list = []

                        # Identifica o canal de índice (Profundidade)
                        index_channel = next((ch for ch in frame.channels if ch.name == frame.index), None)
                        if not index_channel:
                            continue # Frame sem índice (possível dado espúrio)

                        # Extrai Index
                        try:
                            index_vals = index_channel.curves()
                            curves_data[index_channel.name] = index_vals
                        except: continue

                        # Extrai Canais do Frame
                        for ch in frame.channels:
                            if ch.name == frame.index: continue

                            try:
                                # [BEST PRACTICE] Validação de Dimensão
                                # Apenas curvas 1D [1] ou arrays simples.
                                # dlisio retorna dimensão como lista de ints, ex: [1] ou [50, 1]
                                if ch.dimension == [1]:
                                    data = ch.curves()
                                    curves_data[ch.name] = data

                                    # Estatísticas
                                    stats_list.append({
                                        'Mnemonic': ch.name,
                                        'Unit': ch.units,
                                        'Min': np.nanmin(data),
                                        'Max': np.nanmax(data),
                                        'Desc': ch.long_name
                                    })
                            except: pass

                        # Gera DataFrames
                        df_curves = pd.DataFrame(curves_data)
                        df_curves.set_index(index_channel.name, inplace=True)
                        df_stats = pd.DataFrame(stats_list)

                        # 3. Salvamento (Por Frame/LogicalFile)
                        # Adiciona nome do frame ao sufixo para unicidade
                        suffix = f"_{str(f)}_{frame.name}"
                        base_name = file_name + suffix

                        # TXT
                        txt = save_metadata_txt(base_name, output_dir, meta_dict, df_stats)
                        log(f"   [TXT] {os.path.basename(txt)}")

                        # CSV
                        csv = save_curves_csv(base_name, output_dir, df_curves)
                        log(f"   [CSV] {os.path.basename(csv)}")

                        # PNG
                        target_plot_dir = plot_dir if plot_dir else output_dir
                        png = generate_triple_combo(base_name, target_plot_dir, df_curves, None)
                        if png: log(f"   [PNG] {os.path.basename(png)}")

    except Exception as e:
        log(f"   [ERRO] Falha ao processar {file_name}: {e}")

def process_las_multi_output(file_path, output_dir, plot_dir=None):
    """Processa LAS gerando TXT, CSV e PNG."""
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    log(f"\n>>> Processando LAS: {file_name}")

    try:
        las = lasio.read(file_path)

        # --- 1. Metadados ---
        meta_dict = {}
        for item in las.well:
             if item.mnemonic in ['STRT', 'STOP', 'STEP', 'NULL', 'COMP', 'WELL', 'FLD', 'LOC', 'CTRY', 'DATE']:
                 meta_dict[item.mnemonic] = item.value

        # --- 2. Dados ---
        df_curves = las.df()

        stats_list = []
        for curve in las.curves:
             if curve.mnemonic == df_curves.index.name: continue
             try:
                 data = df_curves[curve.mnemonic]
                 stats_list.append({
                    'Mnemonic': curve.mnemonic,
                    'Unit': curve.unit,
                    'Min': data.min(),
                    'Max': data.max(),
                    'Desc': curve.descr
                 })
             except: pass

        df_stats = pd.DataFrame(stats_list)

        # --- 3. Salvamento ---
        txt = save_metadata_txt(file_name, output_dir, meta_dict, df_stats)
        log(f"   [TXT] {os.path.basename(txt)}")

        csv = save_curves_csv(file_name, output_dir, df_curves)
        log(f"   [CSV] {os.path.basename(csv)}")

        target_plot_dir = plot_dir if plot_dir else output_dir
        png = generate_triple_combo(file_name, target_plot_dir, df_curves, None)
        if png: log(f"   [PNG] {os.path.basename(png)}")

    except Exception as e:
        log(f"   [ERRO] Falha ao processar {file_name}: {e}")


# ------------------------------



In [13]:
# @title 🚀 2. Executar Pipeline
import glob

def execute_pipeline():
    log("=== INICIANDO PIPELINE HERMES V2 (Multi-Output) ===")

    # 1. Setup
    if 'setup_workspace_with_credentials' in globals():
        # Configura Log na pasta Output
        PATHS, CRED_PATH = setup_workspace_with_credentials("Analise_Poco_Automatizada")
        # Configura Log na pasta Output
        setup_logger(PATHS['output'])
    else:
        log("[ERRO] Função de setup não definida. Rode a célula 2.")
        return

    # 2. Ingestão
    # Solicita URL ao usuário
    url_input = input("Insira a URL do arquivo DLIS/LAS/ZIP (ou Enter para varrer pasta local): ").strip()

    if 'smart_ingestion' in globals():
        if url_input:
            smart_ingestion(url_input, PATHS['input_run'])
            # Analisa a pasta de download específica
            scan_dir = PATHS['input_run']
        else:
             log("[MODO OFFLINE] Nenhuma URL fornecida. Processando arquivos locais existentes.")
             scan_dir = PATHS['input_root']
    else:
        log("[AVISO] smart_ingestion não definida.")

    # 3. Processamento
    # Lista todos os arquivos na pasta input
    # Define qual diretório escanear (definido acima)
    if 'scan_dir' not in locals(): scan_dir = PATHS['input_root']
    files = glob.glob(os.path.join(scan_dir, "*"))
    log(f"\nArquivos encontrados para processar: {len(files)}")

    for fpath in files:
        ext = os.path.splitext(fpath)[1].lower()
        if ext == '.dlis':
            if 'process_dlis_multi_output' in globals():
                process_dlis_multi_output(fpath, PATHS['output'], PATHS['images'])
        elif ext == '.las':
            if 'process_las_multi_output' in globals():
                 process_las_multi_output(fpath, PATHS['output'], PATHS['images'])

    print("\n=== PIPELINE CONCLUÍDO ===")
    print(f"Arquivos gerados em: {PATHS['output']}")

# Executa
if __name__ == "__main__":
    execute_pipeline()

INFO:HermesPipeline:=== INICIANDO PIPELINE HERMES V2 (Multi-Output) ===
INFO:HermesPipeline:
Verificando estrutura de diretórios em: /content/drive/MyDrive/Analise_Poco_Automatizada
INFO:HermesPipeline:[OK] root: /content/drive/MyDrive/Analise_Poco_Automatizada
INFO:HermesPipeline:[OK] config: /content/drive/MyDrive/Analise_Poco_Automatizada/00_Config
INFO:HermesPipeline:[OK] input_root: /content/drive/MyDrive/Analise_Poco_Automatizada/01_Raw_Data
INFO:HermesPipeline:[CRIADO] input_run: /content/drive/MyDrive/Analise_Poco_Automatizada/01_Raw_Data/Analysis_20260103_192413
INFO:HermesPipeline:[OK] input: /content/drive/MyDrive/Analise_Poco_Automatizada/01_Raw_Data
INFO:HermesPipeline:[CRIADO] output: /content/drive/MyDrive/Analise_Poco_Automatizada/02_Processed/Analysis_20260103_192413
INFO:HermesPipeline:[CRIADO] images: /content/drive/MyDrive/Analise_Poco_Automatizada/03_Plots/Analysis_20260103_192413
INFO:HermesPipeline:
[OK] Credenciais validadas em: /content/drive/MyDrive/Analise_Po


[LOG] Saída redirecionada para: /content/drive/MyDrive/Analise_Poco_Automatizada/02_Processed/Analysis_20260103_192413/pipeline_log_20260103_192413.txt
Insira a URL do arquivo DLIS/LAS/ZIP (ou Enter para varrer pasta local): https://reate.cprm.gov.br/arquivos/index.php/s/DKD0oj9FsZAU8tI/download?path=%2FPOCO%2FCategoria-6%2F6-BRSA-24-AM%2FPerfil%20Convencional&files=6-brsa-24-am_brsa_ait_pex_sdt.dlis


INFO:HermesPipeline:
[INGESTÃO] Conectando a: https://reate.cprm.gov.br/arquivos/index.php/s/DKD0oj9FsZAU8tI/download?path=%2FPOCO%2FCategoria-6%2F6-BRSA-24-AM%2FPerfil%20Convencional&files=6-brsa-24-am_brsa_ait_pex_sdt.dlis
INFO:HermesPipeline:Baixando: 6-brsa-24-am_brsa_ait_pex_sdt.dlis...
INFO:HermesPipeline:Arquivo não é ZIP. Salvando como: 6-brsa-24-am_brsa_ait_pex_sdt.dlis
INFO:HermesPipeline:
Arquivos encontrados para processar: 1
INFO:HermesPipeline:
>>> Processando DLIS: 6-brsa-24-am_brsa_ait_pex_sdt
INFO:HermesPipeline:   [TXT] 6-brsa-24-am_brsa_ait_pex_sdt_LogicalFile(HILTB .017)_1_meta.txt
INFO:HermesPipeline:   [CSV] 6-brsa-24-am_brsa_ait_pex_sdt_LogicalFile(HILTB .017)_1_data.csv
INFO:HermesPipeline:   [PNG] 6-brsa-24-am_brsa_ait_pex_sdt_LogicalFile(HILTB .017)_1_plot.png
INFO:HermesPipeline:   [TXT] 6-brsa-24-am_brsa_ait_pex_sdt_LogicalFile(HILTB .017)_2_meta.txt
INFO:HermesPipeline:   [CSV] 6-brsa-24-am_brsa_ait_pex_sdt_LogicalFile(HILTB .017)_2_data.csv
INFO:HermesPipe


=== PIPELINE CONCLUÍDO ===
Arquivos gerados em: /content/drive/MyDrive/Analise_Poco_Automatizada/02_Processed/Analysis_20260103_192413
